###### ⚡Démarrer Juribot

---

In [ ]:
# %% [code]
from google.colab import drive
import os
import sys
import io


# Configuration des logs (moins bavard pour l'interface)
# os.environ["ASSISTANT_LOG_LEVEL"] = "WARNING"  # Affiche seulement warnings et erreurs
# os.environ["SHOW_LOGS"] = "False"  # Pas d'affichage console

# Capturer la sortie du backend
old_stdout = sys.stdout
sys.stdout = io.StringIO()

In [ ]:


# Monter Google Drive si nécessaire
def mount_google_drive(mount_point='/content/drive', force_remount=False):
    """Monte Google Drive"""
    try:
        if os.path.exists(mount_point) and not force_remount:
            if os.path.exists(os.path.join(mount_point, 'MyDrive')):
                print(f"✅ Drive déjà monté sur {mount_point}")
                return True

        drive.mount(mount_point, force_remount=force_remount)
        print(f"✅ Google Drive monté sur {mount_point}")
        return True
    except Exception as e:
        print(f"❌ Erreur: {str(e)}")
        return False

mount_google_drive()

# Le reste de votre interface...

In [ ]:
# ============================================
# UV-BASED PACKAGE INSTALLER FOR COLAB
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import sys
import subprocess
import importlib
from typing import List, Tuple

def install_missing_packages_with_uv(packages: List[str], verbose: bool = False) -> Tuple[List[str], List[str]]:
    """
    Install only missing packages using uv (ultra-fast).
    
    Args:
        packages: List of package specifications (e.g., 'openai>=1.0.0')
        verbose: Print detailed output
    
    Returns:
        Tuple of (installed_packages, already_installed_packages)
    """
    # Ensure uv is installed
    try:
        subprocess.run(['uv', '--version'], capture_output=True, check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("📦 Installing uv...")
        subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
        print("✅ uv installed")
    
    def get_package_name(pkg_spec: str) -> str:
        """Extract base package name from version spec"""
        # Remove version constraints (>=, <=, ==, etc.)
        for separator in ['>=', '<=', '==', '>', '<', '~=', '!=']:
            if separator in pkg_spec:
                return pkg_spec.split(separator)[0]
        return pkg_spec
    
    def is_package_installed(pkg_name: str) -> bool:
        """Check if package is already installed"""
        # Convert hyphens to underscores for import (common in Python packages)
        import_name = pkg_name.replace('-', '_')
        try:
            importlib.import_module(import_name)
            return True
        except ImportError:
            return False
    
    # Separate packages into installed and missing
    installed = []
    missing = []
    
    print("🔍 Checking installed packages...")
    for pkg_spec in packages:
        pkg_name = get_package_name(pkg_spec)
        if is_package_installed(pkg_name):
            installed.append(pkg_spec)
            if verbose:
                print(f"  ✅ {pkg_name} already installed")
        else:
            missing.append(pkg_spec)
            if verbose:
                print(f"  ❌ {pkg_name} not installed")
    
    # Install missing packages with uv
    if missing:
        print(f"\n📦 Installing {len(missing)} missing packages with uv...")
        
        # Install in batches to avoid command line length limits
        batch_size = 10
        for i in range(0, len(missing), batch_size):
            batch = missing[i:i+batch_size]
            print(f"  Installing batch {i//batch_size + 1}/{(len(missing)-1)//batch_size + 1}...")
            
            cmd = ['uv', 'pip', 'install', '--system', '-q'] + batch
            try:
                subprocess.run(cmd, check=True, capture_output=True, text=True)
            except subprocess.CalledProcessError as e:
                print(f"  ⚠️  Error installing batch: {e.stderr}")
                # Fall back to installing individually
                for pkg in batch:
                    print(f"    Installing {pkg} individually...")
                    subprocess.run(['uv', 'pip', 'install', '--system', '-q', pkg], check=True)
        
        print(f"✅ Installed {len(missing)} packages")
    else:
        print("✅ All packages already installed!")
    
    return missing, installed

def setup_colab_environment(verbose: bool = False):
    """
    Complete Colab environment setup with uv.
    """
    packages = [
        'openai>=1.0.0',
        'anthropic>=0.18.0',
        'groq>=0.4.0',
        'mistralai>=0.1.0',
        'google-generativeai>=0.3.0',
        'pypdf>=3.0.0',
        'tiktoken>=0.5.0',
        'PyPDF2>=3.0.0',
        'python-docx>=1.0.0',
        'ipywidgets>=7.7.0',
        'langchain>=0.3.0',
        'langchain-community>=0.3.0',
        'langchain-core>=0.3.0',
        'langchain-openai>=0.2.0',
        'langchain-anthropic>=0.2.0',
        'langchain-google-genai>=0.1.0',
    ]
    
    print(f"🐍 Python version: {sys.version}")
    print(f"📦 Total packages to check: {len(packages)}")
    
    # Install missing packages
    installed, already = install_missing_packages_with_uv(packages, verbose)
    
    # Install local package
    print("\n📦 Installing your local package...")
    subprocess.run([
        'pip', 'install', '--no-deps', '-e', 
        '/content/drive/MyDrive/assistant-juridique', '-q'
    ], check=True)
    print("✅ Local package installed")
    
    # Return summary
    return {
        'installed': installed,
        'already_installed': already,
        'total_installed': len(installed)
    }

# ============================================
# RUN THE SETUP
# ============================================

# Run with verbose output to see what's happening
result = setup_colab_environment(verbose=True)

print("\n" + "="*50)
print("📊 INSTALLATION SUMMARY")
print("="*50)
print(f"✅ Already installed: {len(result['already_installed'])} packages")
print(f"📦 Newly installed: {len(result['installed'])} packages")
if result['installed']:
    print(f"   - {', '.join(result['installed'][:5])}")
    if len(result['installed']) > 5:
        print(f"   ... and {len(result['installed']) - 5} more")

# Test imports
print("\n🔍 Testing critical imports...")
try:
    from langchain_openai import ChatOpenAI
    from langchain_anthropic import ChatAnthropic
    from langchain_google_genai import ChatGoogleGenerativeAI
    import openai, anthropic, groq, mistralai
    import pypdf, tiktoken, PyPDF2, docx
    import ipywidgets
    print("✅ All critical imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")

print("\n🎉 Environment ready!")

In [ ]:
# Add directly to Python path
import sys
package_path = '/content/drive/MyDrive/assistant-juridique'
if package_path not in sys.path:
    sys.path.insert(0, package_path)
    
from utils.config_llm import get_modele_actif
from utils.launch_interface import launch_interface
from utils.config_clients import LLM_Client
modele_actif, message, MODELES_DISPONIBLES, API_KEYS = get_modele_actif()
llm_client = LLM_Client(modele_actif, API_KEYS, MODELES_DISPONIBLES)

In [ ]:

# Importer et initialiser les logs
# from utils.utils_logging import setup_logging, get_logger
# setup_logging()
# logger = get_logger(__name__)

# logger.info("🚀 Démarrage de l'interface")

In [ ]:
# # Dans le second notebook, exécutez le premier notebook
# # avec gestion des espaces dans le chemin
# %run "/content/drive/MyDrive/assistant-juridique/BACKEND.ipynb"
# # Restaurer la sortie
# sys.stdout = old_stdout
# # logger.info("✅ Backend chargé silencieusement")
# print("✅ Backend chargé silencieusement")

#  🤖 **Prêt à vous libérer du temps**

In [ ]:
launch_interface(llm_client)